# 01 — Data Exploration
Hateful Memes Challenge (Meta AI)

This notebook explores the dataset: class distribution, text statistics, sample images, and embedding space analysis.

In [ ]:
import os, sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from collections import Counter

# ── Environment detection ─────────────────────────────────────────────────
ON_KAGGLE = os.path.exists('/kaggle/input')

if ON_KAGGLE:
    KAGGLE_BASE  = '/kaggle/input/meta-hateful-meme-detection/hateful-meme-project'
    PROJECT_ROOT = KAGGLE_BASE
    sys.path.insert(0, KAGGLE_BASE)
    DATA_DIR     = os.path.join(KAGGLE_BASE, 'data')
    OUTPUT_DIR   = '/kaggle/working/outputs'
else:
    PROJECT_ROOT = os.path.abspath('..')
    sys.path.insert(0, PROJECT_ROOT)
    DATA_DIR     = os.path.join(PROJECT_ROOT, 'data')
    OUTPUT_DIR   = os.path.join(PROJECT_ROOT, 'outputs')

IMAGE_ROOT = os.path.join(DATA_DIR, 'img')
os.makedirs(OUTPUT_DIR, exist_ok=True)

print('Environment :', 'Kaggle' if ON_KAGGLE else 'Local')
print('Project root:', PROJECT_ROOT)
print('Data dir    :', DATA_DIR)


## 1. Load the splits

In [ ]:
def load_jsonl(path):
    return [json.loads(l) for l in open(path, 'r', encoding='utf-8')]

train = load_jsonl(os.path.join(DATA_DIR, 'train.jsonl'))
dev   = load_jsonl(os.path.join(DATA_DIR, 'dev.jsonl'))
test  = load_jsonl(os.path.join(DATA_DIR, 'test.jsonl'))

print(f'Train : {len(train):,}')
print(f'Dev   : {len(dev):,}')
print(f'Test  : {len(test):,}')

df_train = pd.DataFrame(train)
df_dev   = pd.DataFrame(dev)
df_test  = pd.DataFrame(test)
df_train.head()

## 2. Class Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for ax, df, title in zip(axes, [df_train, df_dev], ['Train', 'Dev']):
    counts = df['label'].value_counts().sort_index()
    ax.bar(['Non-hateful (0)', 'Hateful (1)'], counts.values,
           color=['steelblue', 'coral'], edgecolor='black')
    ax.set_title(f'{title} — class distribution', fontsize=13)
    ax.set_ylabel('Count')
    for i, v in enumerate(counts.values):
        ax.text(i, v + 20, str(v), ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, 'outputs', 'class_distribution.png'), dpi=150)
plt.show()

## 3. Text Length Analysis

In [ ]:
df_train['text_len'] = df_train['text'].apply(lambda x: len(x.split()))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histogram by class
for label, color in [(0, 'steelblue'), (1, 'coral')]:
    subset = df_train[df_train['label'] == label]['text_len']
    axes[0].hist(subset, bins=30, alpha=0.6, color=color,
                 label=f'Label {label}')
axes[0].set_xlabel('Word count')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Text length by class (train)')
axes[0].legend()

# Percentiles
p = [50, 75, 90, 95, 99]
vals = np.percentile(df_train['text_len'], p)
axes[1].barh([str(x)+'th' for x in p], vals, color='mediumseagreen')
axes[1].set_xlabel('Word count')
axes[1].set_title('Text length percentiles (train)')

plt.tight_layout()
plt.show()

print(df_train.groupby('label')['text_len'].describe().round(2))

## 4. Sample Memes

In [ ]:
from PIL import Image

def show_memes(samples, n=6, title=''):
    fig, axes = plt.subplots(1, n, figsize=(4*n, 4))
    for ax, s in zip(axes, samples[:n]):
        img_path = os.path.join(IMAGE_ROOT, os.path.basename(s['img']))
        img = Image.open(img_path).convert('RGB')
        ax.imshow(img)
        label_str = 'HATEFUL' if s.get('label', -1) == 1 else 'NOT hateful'
        ax.set_title(f"{label_str}\n\"{s['text'][:50]}…\"" if len(s['text']) > 50
                     else f"{label_str}\n\"{s['text']}\"",
                     fontsize=8, wrap=True)
        ax.axis('off')
    fig.suptitle(title, fontsize=14)
    plt.tight_layout()
    plt.show()

# 6 hateful examples
hateful_samples = df_train[df_train['label'] == 1].sample(6, random_state=42).to_dict('records')
show_memes(hateful_samples, title='Hateful samples')

# 6 non-hateful examples
benign_samples = df_train[df_train['label'] == 0].sample(6, random_state=42).to_dict('records')
show_memes(benign_samples, title='Non-hateful samples')

## 5. CLIP Embedding Space (t-SNE)
Visualise how well CLIP separates hateful vs non-hateful memes.

In [ ]:
import torch
from torch.utils.data import DataLoader
from transformers import CLIPProcessor
from sklearn.manifold import TSNE

from src.datasets.meme_dataset import HatefulMemeDataset
from src.models.clip_encoder import CLIPEncoder

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

processor = CLIPProcessor.from_pretrained('openai/clip-vit-base-patch32')
encoder   = CLIPEncoder(freeze=True).to(device)

# Use a 300-sample subset for speed
subset = df_train.sample(300, random_state=0).to_dict('records')
subset_ds = HatefulMemeDataset(
    jsonl_path=os.path.join(DATA_DIR, 'train.jsonl'),
    image_root=IMAGE_ROOT,
    processor=processor,
    split='train',
)
# Override data to only 300 samples
from src.utils.data_loader import load_jsonl as _load
subset_ds.data = _load(os.path.join(DATA_DIR, 'train.jsonl'))[:300]

loader = DataLoader(subset_ds, batch_size=32, shuffle=False, num_workers=0)

all_img, all_txt, all_labels = [], [], []
encoder.eval()
with torch.no_grad():
    for batch in loader:
        img_e, txt_e = encoder(
            batch['input_ids'].to(device),
            batch['attention_mask'].to(device),
            batch['pixel_values'].to(device),
        )
        all_img.append(img_e.cpu().numpy())
        all_txt.append(txt_e.cpu().numpy())
        all_labels.extend(batch['label'].numpy())

img_feats = np.concatenate(all_img)     # (300, 512)
txt_feats = np.concatenate(all_txt)     # (300, 512)
labels_arr = np.array(all_labels)

print('Feature shapes:', img_feats.shape, txt_feats.shape)

In [ ]:
# Concatenate image+text features then run t-SNE
joint = np.concatenate([img_feats, txt_feats], axis=1)   # (300, 1024)
tsne  = TSNE(n_components=2, perplexity=30, random_state=42, n_iter=1000)
emb2d = tsne.fit_transform(joint)

fig, ax = plt.subplots(figsize=(8, 6))
colors = {0: 'steelblue', 1: 'coral'}
for lbl, col in colors.items():
    mask = labels_arr == lbl
    ax.scatter(emb2d[mask, 0], emb2d[mask, 1],
               c=col, label=['Non-hateful', 'Hateful'][lbl],
               alpha=0.7, s=30)

ax.set_title('t-SNE of CLIP image+text embeddings (300 train samples)', fontsize=13)
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, 'outputs', 'tsne_clip_embeddings.png'), dpi=150)
plt.show()

## 6. CLIP Zero-Shot Baseline
Measure how well CLIP's similarity score alone separates the classes (no fine-tuning).

In [ ]:
from sklearn.metrics import roc_auc_score, accuracy_score

# CLIP assigns higher similarity between image and its text → use as score
# For zero-shot: cosine(img, txt) as a proxy
cos_sim = (img_feats * txt_feats).sum(axis=1)  # (300,) already L2-norm
valid = labels_arr != -1

auroc_zs = roc_auc_score(labels_arr[valid], cos_sim[valid])
print(f'Zero-shot CLIP cosine similarity AUROC: {auroc_zs:.4f}')
print('(random = 0.50, a score anywhere near 0.50 means CLIP alone is insufficient)')